In [ ]:
import pandas as pd
url = 'https://drive.google.com/drive/folders/1mHBDnFvMOgxnZIVSAw1LyT8b2-qbzB8T?usp=sharing'
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]


# Data cleaning

In [ ]:
brands = pd.read_csv("/content/brands.csv")
brands_df = brands.copy()
orderlines = pd.read_csv("/content/orderlines.csv")
orderlines_df = orderlines.copy()
orders = pd.read_csv("/content/orders.csv")
orders_df = orders.copy()
products = pd.read_csv("/content/products.csv")
products_df = products.copy()

## Orders

### Missing values

In [ ]:
# The "total_paid" column has 5 null values (since they're not too much, we delete them)
orders_df = orders_df.dropna(subset=["total_paid"])

### Convert "created-date" column in datetime

In [ ]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### Clean "State" column with .strip()

In [ ]:
orders_df["state"] = orders_df["state"].str.strip()

## Orderlines

### Clean "unit-price" column

In [ ]:
# 1. Convert prices to a temporary string format to safely perform checks
unit_price_str = orderlines_df["unit_price"].astype(str)

# 2. Mask: Identify corrupted prices following the specific pattern nn.nnn.nnn
mask_wrong_prices = unit_price_str.str.contains(r"^\d{2}\.\d{3}\.\d{3}$", na=False)

# 3. Data cleaning: Remove the first dot only from prices with multiple dots
orderlines_df["unit_price"] = unit_price_str.str.replace(r"\.(?=.*\.)", "", regex=True)

# 4. Data conversion: Transform the cleaned strings into numeric float values
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"], errors="coerce")

# 5. Surgical fix: Divide targeted wrong prices by 10 using the initial mask
orderlines_df.loc[mask_wrong_prices, "unit_price"] = orderlines_df.loc[mask_wrong_prices, "unit_price"] / 10

# 6. Final adjustment: Round all values to standard 2 decimal places
orderlines_df["unit_price"] = orderlines_df["unit_price"].round(2)

# 7. Filter: Keep only the rows where "unit_price" is greater than 0
orderlines_df = orderlines_df[orderlines_df["unit_price"] > 0]

orderlines_df = orderlines_df[orderlines_df["unit_price"] <= 50000]


### Convert "date" column in datetime

In [ ]:
# Convert text strings to proper datetime format in the "date" column
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

### Change "id_order" column in "order_id"

In [ ]:
# Change the name of the "id_order" column in "order_id"
orderlines_df = orderlines_df.rename(columns = {"id_order" : "order_id"})

### Delete the product_id column

In [ ]:
# Drop this column because is not important
orderlines_df = orderlines_df.drop(columns= "product_id")

### Clean "sku" column with strip.()

In [ ]:
orderlines_df["sku"] = orderlines_df["sku"].str.strip()

## Products

### Delete all duplicates

In [ ]:
# Remove completely identical duplicate rows from the DataFrame
products_df.drop_duplicates(inplace=True)

# Remove any remaining rows with duplicate SKUs, keeping only the first occurrence
products_df = products_df[~products_df["sku"].duplicated(keep="first")]

### Clean the "Price" column

In [ ]:
# 1. Convert prices to string format for safety during checks
prices_prod_str = products_df["price"].astype(str)

# 2. Mask: Find all the prices with the wrong format nn.nnn.nnn
mask_prod_wrong = prices_prod_str.str.contains(r"^\d{2}\.\d{3}\.\d{3}$", na=False)

# 3. Remove the first dot only from prices with multiple dots
products_df["price"] = prices_prod_str.str.replace(r"\.(?=.*\.)", "", regex=True)

# 4. Convert prices to numeric float for mathematical operations
products_df["price"] = pd.to_numeric(products_df["price"], errors="coerce")

# 5. Apply the surgical fix: divide targeted wrong prices by 10
products_df.loc[mask_prod_wrong, "price"] = products_df.loc[mask_prod_wrong, "price"] / 10

# 6. Round to 2 decimals and drop the missing values
products_df["price"] = products_df["price"].round(2)

products_df = products_df[products_df["price"] <= 50000]

### Manage the missing values in "Desc", "Type", "Price" columns

In [ ]:
# Fill missing product descriptions with a placeholder string to avoid null values
products_df["desc"] = products_df["desc"].fillna(products_df["name"])


# Assign a placeholder category name to products that are missing a type code
products_df["type"] = products_df["type"].fillna("Unknown")

# Drop the 46 missing values to ensure catalog integrity
products_df = products_df.dropna(subset=["price"])


### Clean all the object columns with .strip()

In [ ]:
products_df["sku"] = products_df["sku"].str.strip()

products_df["name"] = products_df["name"].str.strip()

products_df["desc"] = products_df["desc"].str.strip()

products_df["type"] = products_df["type"].str.strip()

### Remove the "Promo_price" column

In [ ]:
products_df = products_df.drop(columns = ["promo_price"])

## Brands

### Clean "Short" and "Long" columns

In [ ]:
# Remove hidden whitespaces from brand names
brands_df["long"] = brands_df["long"].str.strip()

# Remove hidden whitespaces from the 3-character brand code
brands_df["short"] = brands_df["short"].str.strip()

# Drop duplicate brand codes to ensure uniqueness before merging
brands_df = brands_df[~brands_df["short"].duplicated(keep="first")]